# Data Cleaning

This notebook reproduces the conservative telecom cleaning logic implemented in `scripts/etl_pipeline.py`.


In [ ]:
import pandas as pd
import re

raw_path = "data/raw/iranian-telecom-churn.csv"
df = pd.read_csv(raw_path)
print(df.shape)


## Standardize Column Names


In [ ]:
def standardize_name(column_name):
    name = column_name.strip().lower()
    name = re.sub(r"[\s\-]+", "_", name)
    name = re.sub(r"[^a-z0-9_]", "", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

df.columns = [standardize_name(column) for column in df.columns]
df.columns.tolist()


## Remove Exact Duplicates and Validate Types


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

int_columns = [
    "id",
    "subscription_length",
    "charge_amount",
    "seconds_of_use",
    "frequency_of_use",
    "frequency_of_sms",
    "distinct_called_numbers",
    "call_failures",
    "churn",
]
for column in int_columns:
    df[column] = pd.to_numeric(df[column], errors="raise").astype("int64")

for column in ["tariff_plan", "status", "age_group", "complaints"]:
    df[column] = df[column].astype("string").str.strip()

df.dtypes


## Standardize Categories


In [ ]:
df["tariff_plan"] = df["tariff_plan"].str.lower().map({"a": "A", "b": "B"}).astype("string")
df["status"] = df["status"].str.lower().map({"active": "Active", "inactive": "Inactive"}).astype("string")
df["complaints"] = df["complaints"].str.lower().map({"y": "Y", "n": "N"}).astype("string")
df["age_group"] = df["age_group"].astype("string")

for column in ["tariff_plan", "status", "age_group", "complaints"]:
    print(column, sorted(df[column].dropna().unique().tolist()))


## Validation Checks


In [ ]:
print(df.shape)
print()
print(df.isnull().sum())
print()
print(df.duplicated().sum())
print()
print(df["churn"].value_counts(dropna=False))


In [ ]:
df.groupby("churn").mean(numeric_only=True)


## Save Cleaned Output


In [ ]:
processed_path = "data/processed/cleaned_data.csv"
df.to_csv(processed_path, index=False)
print(processed_path)
